In [10]:
# import libraries
import pandas as pd
import numpy as np


TRANSACTION_PATH = pd.read_csv("../data/raw/train_transaction.csv")
IDENTITY_PATH = pd.read_csv("../data/raw/train_identity.csv")

transactions = TRANSACTION_PATH.copy()
identity = IDENTITY_PATH.copy()

In [11]:
print("Data Structure of TRANSACTION_PATH:")
transactions.info()
print("Transaction shape:", transactions.shape)
print()

print("Data Structure of IDENTITY_PATH:")
identity.info()
print("Identity shape:", identity.shape)



Data Structure of TRANSACTION_PATH:
<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), str(14)
memory usage: 1.7 GB
Transaction shape: (590540, 394)

Data Structure of IDENTITY_PATH:
<class 'pandas.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 41 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionID  144233 non-null  int64  
 1   id_01          144233 non-null  float64
 2   id_02          140872 non-null  float64
 3   id_03          66324 non-null   float64
 4   id_04          66324 non-null   float64
 5   id_05          136865 non-null  float64
 6   id_06          136865 non-null  float64
 7   id_07          5155 non-null    float64
 8   id_08          5155 non-null    float64
 9   id_09          74926 non-null   float64
 10  id_10          74926 non-null   float64
 11  id_11          140978 non-null  flo

In [16]:
print("transactions['isFraud'] value counts:")
print(transactions["isFraud"].value_counts())

print("transactions['isFraud'] value counts (normalized):")
print(transactions["isFraud"].value_counts(normalize=True))

print("transactions['TransactionAmt'] descriptive statistics:")
print(transactions["TransactionAmt"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99, 0.999]
))


transactions['isFraud'] value counts:
isFraud
0    569877
1     20663
Name: count, dtype: int64
transactions['isFraud'] value counts (normalized):
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64
transactions['TransactionAmt'] descriptive statistics:
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
50%          68.769000
75%         125.000000
90%         275.293000
95%         445.000000
99%        1104.000000
99.9%      2769.807320
max       31937.391000
Name: TransactionAmt, dtype: float64


In [ ]:

print("transactions['TransactionDT'] descriptive statistics:")
print(transactions["TransactionDT"].describe())

print("First few rows of the dataset:")
print(transactions[["TransactionDT", "TransactionAmt", "isFraud"]].head())



transactions['TransactionDT'] descriptive statistics:
count    5.905400e+05
mean     7.372311e+06
std      4.617224e+06
min      8.640000e+04
25%      3.027058e+06
50%      7.306528e+06
75%      1.124662e+07
max      1.581113e+07
Name: TransactionDT, dtype: float64
First few rows of the dataset:
   TransactionDT  TransactionAmt  isFraud
0          86400            68.5        0
1          86401            29.0        0
2          86469            59.0        0
3          86499            50.0        0
4          86506            50.0        0
Unique values in selected columns:
card4                4
card6                4
ProductCD            5
P_emaildomain       59
R_emaildomain       60
addr2               74
card3              114
card5              119
addr1              332
card2              500
card1            13553
dtype: int64


In [19]:
print("Unique values in selected columns:")
columns = [
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "ProductCD",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain"
]

print(transactions[columns].nunique().sort_values())


Unique values in selected columns:
card4                4
card6                4
ProductCD            5
P_emaildomain       59
R_emaildomain       60
addr2               74
card3              114
card5              119
addr1              332
card2              500
card1            13553
dtype: int64


In [18]:

print("Descriptive statistics for 'card1' value counts:")
print(transactions["card1"].value_counts().describe())

print("Value counts for 'ProductCD':")
print(transactions["ProductCD"].value_counts())


identity_ids = set(identity["TransactionID"])

transactions["has_identity"] = transactions["TransactionID"].isin(identity_ids)

print("Proportion of transactions with identity information:")
print(transactions["has_identity"].value_counts(normalize=True))


print("Fraud rates by identity information:")
print(transactions.groupby("has_identity")["isFraud"].agg(
    ["count", "mean"]
))

Descriptive statistics for 'card1' value counts:
count    13553.000000
mean        43.572641
std        329.084462
min          1.000000
25%          1.000000
50%          4.000000
75%         14.000000
max      14932.000000
Name: count, dtype: float64
Value counts for 'ProductCD':
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64
Proportion of transactions with identity information:
has_identity
False    0.755761
True     0.244239
Name: proportion, dtype: float64
Fraud rates by identity information:
               count      mean
has_identity                  
False         446307  0.020939
True          144233  0.078470


In [22]:
# 1. Fraud rate by ProductCD
print("Fraud rates by ProductCD:")
print(transactions.groupby("ProductCD")["isFraud"].agg(
    ["count", "mean"]
).sort_values("count", ascending=False))

# 2. Fraud rate by card6
print("Fraud rates by card6:")
print(transactions.groupby("card6")["isFraud"].agg(
    ["count", "mean"]
).sort_values("count", ascending=False))



Fraud rates by ProductCD:
            count      mean
ProductCD                  
W          439670  0.020399
C           68519  0.116873
R           37699  0.037826
H           33024  0.047662
S           11628  0.058996
Fraud rates by card6:
                  count      mean
card6                            
debit            439938  0.024263
credit           148986  0.066785
debit or credit      30  0.000000
charge card          15  0.000000


In [21]:
print("Fraud rates by card4:")
print(transactions.groupby("card4")["isFraud"].agg(
    ["count", "mean"]
).sort_values("count", ascending=False))


print("Transaction amounts by fraud status:")
print(transactions.groupby("isFraud")["TransactionAmt"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
))

Fraud rates by card4:
                   count      mean
card4                             
visa              384767  0.034756
mastercard        189217  0.034331
american express    8328  0.028698
discover            6651  0.077282
Transaction amounts by fraud status:
            count        mean         std    min   50%    75%      90%    95%  \
isFraud                                                                         
0        569877.0  134.511665  239.395078  0.251  68.5  120.0  267.112  435.0   
1         20663.0  149.244779  232.212163  0.292  75.0  161.0  335.000  500.0   

            99%        max  
isFraud                     
0        1104.0  31937.391  
1         994.0   5191.000  
